# Scenario: The "Time-Traveler" Bug

In [5]:
import pandas as pd
from sqlalchemy import create_engine
from IPython.display import display

# creating the dataset with logical errors
stay_data = {
    "visit_id": [7001, 7002, 7003, 7004, 7005],
    "patient_id": ["P-10", "P-11", "P-12", "P-13", "P-14"],
    "admit_date": ["2026-05-01", "2026-05-03", "2026-05-10", "2026-05-12", "2026-05-14"],
    "discharge_date": ["2026-05-05", "2026-05-02", "2026-05-15", "2026-05-10", "2026-05-20"]   
    # look at 7002 and 7004 carefully
}
# adding dataset to DataFrame
df_stayData = pd.DataFrame(stay_data)

# Use SQLAlchemy engine instead of sqlite3.connect
engine = create_engine("sqlite:///:memory:")
df_stayData.to_sql("hospital_stay", engine, index = False, if_exists = "replace")
print("*************************** Day 13 Temporal Logic Database is ready!!! **************")
print()

def run_query(query):
    return pd.read_sql_query(query, engine)

*************************** Day 13 Temporal Logic Database is ready!!! **************



# Detecting Logic Violations

In [6]:
# query for all dataset to review the whole data
all_data = "SELECT * FROM hospital_stay"
print("*********************** all data to review ***********")
display(run_query(all_data))
print()
# query to find the visit_id and patient_id for all records where the discharge_date is less than (<) the admit_date
logic_voilation = """
SELECT visit_id, patient_id FROM hospital_stay
WHERE discharge_date < admit_date
"""
print('****************************** logic voilation has been found *******************')
display(run_query(logic_voilation))

*********************** all data to review ***********


,visit_id,patient_id,admit_date,discharge_date
0,7001,P-10,2026-05-01,2026-05-05
1,7002,P-11,2026-05-03,2026-05-02
2,7003,P-12,2026-05-10,2026-05-15
3,7004,P-13,2026-05-12,2026-05-10
4,7005,P-14,2026-05-14,2026-05-20



****************************** logic voilation has been found *******************


,visit_id,patient_id
0,7002,P-11
1,7004,P-13


# Calculating "Length of Stay" (LOS)

In [10]:
# query that calculates the number of days between admission and discharge for the valid records.
# by finding the length of days stayed, we can eleminate the illogic admit and discharge dates 
lenth_of_stay = """
SELECT visit_id, (julianday(discharge_date) - julianday(admit_date))
AS length_of_stay
FROM hospital_stay 
WHERE discharge_date>=admit_date
"""
print('************************** number of valid stayed days *******************')
display(run_query(lenth_of_stay))

************************** number of valid stayed days *******************


,visit_id,length_of_stay
0,7001,4.0
1,7003,5.0
2,7005,6.0
